# Figure 5 — Gene expression enrichment of epicenters

- Extract Allen Human Brain Atlas gene expression for the DK68 parcellation.
- Compare gene expression between **epicenter** and **non-epicenter** regions (ASD-risk gene enrichment).
- Rank all 34 candidate seed regions by their ASD association.
- Render cortical maps of gene-expression correlation and scatter plots of enrichment.

## Part 1 — Extract Allen gene expression for DK68

In [ ]:
"""Extract region-wise gene expression for the Desikan-Killiany (DK68) atlas
from a local copy of the Allen Human Brain Atlas microarray data."""
import abagen
import pandas as pd
import os

OUTPUT_DIR = 'data'
OUTPUT_FILE = 'data/allen_gene_expression_dk68.csv'
LOCAL_DATA_DIR = '/path/to/abagen-data/microarray/'   # <-- set local Allen data dir
os.makedirs(OUTPUT_DIR, exist_ok=True)


def generate_dk68_gene_expression():
    """Generate the DK68 gene-expression matrix from local Allen data."""
    print("=" * 60)
    print("Generating DK68 gene-expression matrix from local Allen Brain Atlas data")
    print("=" * 60)
    print(f"Local data dir: {LOCAL_DATA_DIR}")

    # Verify local donor data
    expected_donors = ['9861', '10021', '12876', '14380', '15496', '15697']
    all_present = True
    for donor_id in expected_donors:
        donor_dir = os.path.join(LOCAL_DATA_DIR, f'normalized_microarray_donor{donor_id}')
        present = os.path.exists(donor_dir)
        print(f"  {'OK ' if present else 'MISSING'} donor{donor_id}")
        all_present = all_present and present
    if not all_present:
        print("Error: local Allen data is incomplete; aborting.")
        return None

    # Fetch the DK68 atlas
    atlas = abagen.datasets.fetch_desikan_killiany()
    atlas_info = pd.read_csv(atlas['info'])
    print(f"Atlas: {len(atlas_info)} regions")

    # Extract expression from local data (donor-averaged matrix)
    expression = abagen.get_expression_data(
        atlas['image'], atlas['info'],
        data_dir=LOCAL_DATA_DIR,   # use the local data directory
        region_agg='donors',       # aggregate across donors
        missing='centroids',       # missing data -> region centroids
        return_donors=False,       # return donor-averaged matrix
        n_proc=1,
        verbose=1,
    )
    print(f"Result: {expression.shape[0]} regions x {expression.shape[1]} genes, "
          f"NaN = {expression.isna().sum().sum()}")

    expression.index.name = 'Region_ID'
    expression.to_csv(OUTPUT_FILE)
    print(f"Saved to {OUTPUT_FILE} ({os.path.getsize(OUTPUT_FILE)/1e6:.2f} MB)")
    return expression


if __name__ == '__main__':
    result = generate_dk68_gene_expression()
    if result is not None:
        print(f"Done: {result.shape[1]} genes x {result.shape[0]} regions")

## Panel A — MIND-ASD gene association across all 34 left-hemisphere seeds

For each seed (Subtype L), correlate its MIND connectivity fingerprint against the spatial expression profile of every gene; compare epicenter vs non-epicenter seeds and rank all seeds.

In [ ]:
"""
Panel A: comprehensive MIND-ASD gene association analysis over all 34
left-hemisphere cortical seed regions.
  - For each seed (Subtype L), correlate its MIND connectivity fingerprint
    against the spatial expression profile of every gene.
  - Filter high-confidence SFARI ASD genes.
  - Compare epicenter vs non-epicenter regions and rank all seeds.
"""
import os
import shutil
import numpy as np
import pandas as pd
import abagen
from scipy.stats import pearsonr, mannwhitneyu
from statsmodels.stats.multitest import multipletests
from tqdm import tqdm

# --- Configuration ---
METADATA_PATHS = {
    'ABIDE2': 'data/ABIDE2_AGE_Sub.xlsx',
    'CABIC': 'data/CABIC_AGE_Sub.xlsx',
}
PATH_COL_NAME = 'aparc'
SUBTYPE_COL = 'SUBTYPE_LABEL'
TARGET_SUBTYPE = 0

GENE_CACHE_FILE = 'data/allen_gene_expression_dk68.csv'
SFARI_FILE_PATH = 'data/SFARI-Gene_genes_05-01-2026release_06-07-2026export.csv'
OUTPUT_DIR = 'output/All34Seeds_MIND_Gene_Analysis'
FIG5_BACKUP_DIR = 'Fig5/SubA'
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(FIG5_BACKUP_DIR, exist_ok=True)

# The 5 core epicenter seed regions used for comparison
EPICENTER_SEEDS = ['temporalpole', 'caudalanteriorcingulate', 'insula', 'frontalpole', 'cuneus']


def get_mind_connectivity_vector(excel_path, target_subtype, seed_region):
    """Return the subtype-average MIND connectivity fingerprint of a seed region."""
    df = pd.read_excel(excel_path)
    df_sub = df[df[SUBTYPE_COL] == target_subtype]
    if len(df_sub) == 0:
        return None, None
    dk_atlas_info = abagen.datasets.fetch_desikan_killiany()
    atlas_meta = pd.read_csv(dk_atlas_info['info'])
    left_meta = atlas_meta[(atlas_meta['structure'] == 'cortex') & (atlas_meta['hemisphere'] == 'L')]
    seed_row = left_meta[left_meta['label'] == seed_region]
    if seed_row.empty:
        return None, None
    seed_idx = left_meta.index.get_loc(seed_row.index[0])
    matrices = []
    for _, row in df_sub.iterrows():
        path = row[PATH_COL_NAME]
        if pd.isna(path) or not os.path.exists(path):
            continue
        try:
            matrices.append(pd.read_csv(path, sep=None, engine='python').values)
        except Exception:
            continue
    if not matrices:
        return None, None
    mean_matrix = np.mean(matrices, axis=0)
    return mean_matrix[seed_idx, :], seed_idx


def filter_high_confidence_asd_genes():
    """Select high-confidence ASD genes from the SFARI Gene list."""
    try:
        df_sfari = pd.read_csv(SFARI_FILE_PATH)
        high_conf = df_sfari[
            (df_sfari["gene-score"] == 1)
            & (df_sfari["number-of-reports"] >= 100)
            & (df_sfari["genetic-category"].str.contains("Rare Single Gene Mutation|Functional", na=False))
        ].copy()
        high_conf = high_conf.sort_values("number-of-reports", ascending=False).reset_index(drop=True)
        asd_core_genes = high_conf["gene-symbol"].tolist()
        print(f"High-confidence SFARI ASD genes: {len(asd_core_genes)}")
        print(f"Genes: {asd_core_genes}")
        return asd_core_genes
    except Exception as e:
        print(f"Gene filtering failed: {e}")
        return []


def calculate_gene_connectivity_correlation(X_z, Y_z, gene_symbols):
    """Pearson correlation between each gene's spatial expression and the connectivity vector."""
    n_genes = X_z.shape[1]
    r_values, p_values = np.zeros(n_genes), np.zeros(n_genes)
    Y_flat = Y_z.flatten()
    for i in range(n_genes):
        r_values[i], p_values[i] = pearsonr(X_z[:, i], Y_flat)
    df_res = pd.DataFrame({'GeneSymbol': gene_symbols, 'Spatial_R': r_values, 'Raw_P': p_values})
    _, df_res['FDR_P'], _, _ = multipletests(df_res['Raw_P'], method='fdr_bh')
    return df_res


def analyze_single_seed(center, excel_path, seed_region, expression_df, gene_symbols, left_meta, asd_genes):
    """Correlate a seed region's MIND fingerprint against all gene expression profiles."""
    connectivity_vector, seed_idx = get_mind_connectivity_vector(excel_path, TARGET_SUBTYPE, seed_region)
    if connectivity_vector is None:
        return None
    valid_y, valid_X = [], []
    for idx, (_, row) in enumerate(left_meta.iterrows()):
        region_id = int(row['id'])
        if region_id in expression_df.index:
            valid_X.append(expression_df.loc[region_id].values)
            matrix_idx = left_meta.index.get_loc(left_meta.index[idx])
            valid_y.append(connectivity_vector[matrix_idx])
    X = np.array(valid_X)
    Y = np.array(valid_y).reshape(-1, 1)
    X_z = (X - np.mean(X, axis=0)) / (np.std(X, axis=0) + 1e-10)
    Y_z = (Y - np.mean(Y)) / (np.std(Y) + 1e-10)
    df_full = calculate_gene_connectivity_correlation(X_z, Y_z, gene_symbols)
    gene_results = {}
    for gene in asd_genes:
        gene_row = df_full[df_full['GeneSymbol'] == gene]
        if not gene_row.empty:
            gene_results[gene] = {'r': gene_row['Spatial_R'].iloc[0],
                                  'p_raw': gene_row['Raw_P'].iloc[0],
                                  'p_fdr': gene_row['FDR_P'].iloc[0]}
        else:
            gene_results[gene] = {'r': np.nan, 'p_raw': 1.0, 'p_fdr': 1.0}
    n_sig = len(df_full[df_full['FDR_P'] < 0.05])
    asd_gene_rs = [gene_results[g]['r'] for g in asd_genes if not np.isnan(gene_results[g]['r'])]
    mean_asd_r = np.mean(asd_gene_rs) if asd_gene_rs else 0
    return {'seed_region': seed_region, 'gene_results': gene_results, 'n_regions': len(valid_y),
            'n_sig_genes': n_sig, 'mean_asd_gene_r': mean_asd_r, 'full_results': df_full}


def compare_epicenter_vs_nonepicenters(all_seed_results, asd_genes, center):
    """Compare ASD-gene associations between epicenter and non-epicenter seeds."""
    print(f"\n{'='*80}")
    print(f"Epicenter vs non-epicenter comparison - {center}")
    print(f"{'='*80}")
    epicenter_results = [r for r in all_seed_results if r['seed_region'] in EPICENTER_SEEDS]
    non_epicenter_results = [r for r in all_seed_results if r['seed_region'] not in EPICENTER_SEEDS]
    comparison_data = []
    for gene in asd_genes:
        epicenter_rs = [r['gene_results'][gene]['r'] for r in epicenter_results
                        if gene in r['gene_results'] and not np.isnan(r['gene_results'][gene]['r'])]
        non_epicenter_rs = [r['gene_results'][gene]['r'] for r in non_epicenter_results
                            if gene in r['gene_results'] and not np.isnan(r['gene_results'][gene]['r'])]
        epi_mean = np.mean(epicenter_rs) if epicenter_rs else np.nan
        epi_std = np.std(epicenter_rs) if epicenter_rs else np.nan
        non_epi_mean = np.mean(non_epicenter_rs) if non_epicenter_rs else np.nan
        non_epi_std = np.std(non_epicenter_rs) if non_epicenter_rs else np.nan
        if len(epicenter_rs) >= 2 and len(non_epicenter_rs) >= 2:
            _, p_val = mannwhitneyu(epicenter_rs, non_epicenter_rs, alternative='two-sided')
        else:
            p_val = 1.0
        comparison_data.append({'Gene': gene, 'Epicenter_Mean': epi_mean, 'Epicenter_Std': epi_std,
                                'NonEpicenter_Mean': non_epi_mean, 'NonEpicenter_Std': non_epi_std,
                                'Difference': epi_mean - non_epi_mean, 'P_Value': p_val})
    df_compare = pd.DataFrame(comparison_data)
    if df_compare['P_Value'].dropna().shape[0] > 0:
        _, fdr_p, _, _ = multipletests(df_compare['P_Value'].fillna(1), method='fdr_bh')
        df_compare['FDR_P'] = fdr_p
        df_compare['Sig'] = df_compare['FDR_P'].apply(
            lambda x: '***' if x < 0.001 else ('**' if x < 0.01 else ('*' if x < 0.05 else 'n.s.')))
    else:
        df_compare['FDR_P'] = 1.0
        df_compare['Sig'] = 'n.s.'
    print(df_compare[['Gene', 'Epicenter_Mean', 'NonEpicenter_Mean', 'Difference', 'P_Value', 'FDR_P', 'Sig']].to_string(index=False))

    epi_n_sigs = [r['n_sig_genes'] for r in epicenter_results]
    non_epi_n_sigs = [r['n_sig_genes'] for r in non_epicenter_results]
    epi_mean_rs = [r['mean_asd_gene_r'] for r in epicenter_results]
    non_epi_mean_rs = [r['mean_asd_gene_r'] for r in non_epicenter_results]
    print(f"\nMean # significant genes (FDR<0.05): epicenter={np.mean(epi_n_sigs):.1f}, "
          f"non-epicenter={np.mean(non_epi_n_sigs):.1f}")
    print(f"Mean ASD-gene r: epicenter={np.mean(epi_mean_rs):.3f}, non-epicenter={np.mean(non_epi_mean_rs):.3f}")
    if len(epi_n_sigs) >= 2 and len(non_epi_n_sigs) >= 2:
        _, p_n_sig = mannwhitneyu(epi_n_sigs, non_epi_n_sigs, alternative='two-sided')
        _, p_mean_r = mannwhitneyu(epi_mean_rs, non_epi_mean_rs, alternative='two-sided')
        print(f"  U-tests: p(#sig genes)={p_n_sig:.4f}, p(mean r)={p_mean_r:.4f}")
    compare_path = os.path.join(OUTPUT_DIR, f'{center}_Epicenter_vs_NonEpicenter_Comparison.csv')
    df_compare.to_csv(compare_path, index=False)
    print(f"Comparison saved: {compare_path}")
    return df_compare


def rank_all_seeds_by_asd_association(all_seed_results, center, output_dir):
    """Rank all seed regions by their ASD-gene association strength."""
    print(f"\n{'='*80}")
    print(f"All-seed ranking - {center}")
    print(f"{'='*80}")
    df_rank = pd.DataFrame([
        {'Seed_Region': r['seed_region'],
         'Is_Epicenter': 'Yes' if r['seed_region'] in EPICENTER_SEEDS else 'No',
         'N_Sig_Genes': r['n_sig_genes'],
         'Mean_ASD_Gene_R': r['mean_asd_gene_r']} for r in all_seed_results])
    df_by_r = df_rank.sort_values('Mean_ASD_Gene_R', ascending=False).reset_index(drop=True)
    df_by_r['Rank_by_R'] = range(1, len(df_by_r) + 1)
    df_by_n = df_rank.sort_values('N_Sig_Genes', ascending=False).reset_index(drop=True)
    df_by_n['Rank_by_N'] = range(1, len(df_by_n) + 1)
    df_final = pd.merge(df_by_r[['Seed_Region', 'Rank_by_R']],
                        df_by_n[['Seed_Region', 'Rank_by_N', 'N_Sig_Genes', 'Mean_ASD_Gene_R', 'Is_Epicenter']],
                        on='Seed_Region')
    print(df_final.head(10).to_string(index=False))
    rank_path = os.path.join(output_dir, f'{center}_All_Seeds_Ranking.csv')
    df_final.to_csv(rank_path, index=False)
    print(f"Ranking saved: {rank_path}")
    return df_final


def run_all34seeds_analysis():
    """Run the full 34-seed analysis for both cohorts."""
    print("=" * 80)
    print("  MIND-ASD gene association across all 34 left-hemisphere seeds (Subtype L)")
    print("=" * 80)
    ASD_CORE_GENES = filter_high_confidence_asd_genes()
    if not ASD_CORE_GENES:
        return
    expression_df = pd.read_csv(GENE_CACHE_FILE, index_col=0)
    gene_symbols = expression_df.columns.tolist()
    ASD_CORE_GENES = [g for g in ASD_CORE_GENES if g in gene_symbols]
    if not ASD_CORE_GENES:
        print("No usable core genes; aborting.")
        return
    dk_atlas_info = abagen.datasets.fetch_desikan_killiany()
    atlas_meta = pd.read_csv(dk_atlas_info['info'])
    left_meta = atlas_meta[(atlas_meta['structure'] == 'cortex') & (atlas_meta['hemisphere'] == 'L')]
    all_34_seeds = left_meta['label'].tolist()
    print(f"\nAnalyzing {len(all_34_seeds)} left-hemisphere seeds; epicenters: {EPICENTER_SEEDS}")

    for center, excel_path in METADATA_PATHS.items():
        print(f"\n{'='*80}\nProcessing {center}\n{'='*80}")
        seed_results = []
        for seed in tqdm(all_34_seeds, desc=f"{center} - analyzing seeds"):
            result = analyze_single_seed(center, excel_path, seed, expression_df, gene_symbols, left_meta, ASD_CORE_GENES)
            if result:
                seed_results.append(result)
        print(f"Successfully analyzed {len(seed_results)} seeds")

        # Save detailed per-seed results
        detailed_results = []
        for result in seed_results:
            row_data = {'Seed_Region': result['seed_region'],
                        'Is_Epicenter': 'Yes' if result['seed_region'] in EPICENTER_SEEDS else 'No',
                        'N_Regions_Analyzed': result['n_regions'],
                        'N_Sig_Genes_Total': result['n_sig_genes'],
                        'Mean_ASD_Gene_R': result['mean_asd_gene_r']}
            for gene in ASD_CORE_GENES:
                if gene in result['gene_results']:
                    row_data[f'{gene}_R'] = result['gene_results'][gene]['r']
                    row_data[f'{gene}_P'] = result['gene_results'][gene]['p_fdr']
            detailed_results.append(row_data)
        df_detailed = pd.DataFrame(detailed_results)
        detailed_path = os.path.join(OUTPUT_DIR, f'{center}_All34Seeds_Detailed_Results.csv')
        df_detailed.to_csv(detailed_path, index=False)
        shutil.copy2(detailed_path, os.path.join(FIG5_BACKUP_DIR, f'SubA_{center}_Gene.csv'))
        print(f"Detailed results saved: {detailed_path} -> Fig5/SubA/SubA_{center}_Gene.csv")

        compare_epicenter_vs_nonepicenters(seed_results, ASD_CORE_GENES, center)
        df_rank = rank_all_seeds_by_asd_association(seed_results, center, OUTPUT_DIR)
        shutil.copy2(os.path.join(OUTPUT_DIR, f'{center}_All_Seeds_Ranking.csv'),
                     os.path.join(FIG5_BACKUP_DIR, f'{center}_All_Seeds_Ranking.csv'))


if __name__ == '__main__':
    run_all34seeds_analysis()

## Step 2 — Cortical maps of core-gene spatial correlations

In [ ]:
"""Step 2: render cortical maps of the spatial gene-correlation values for the
10 core ASD genes (left-hemisphere views, 5x2 grid with a shared colorbar)."""
import os
os.environ['DISPLAY'] = ':99'
os.environ['__GLX_VENDOR_LIBRARY_NAME'] = 'nvidia'

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
import matplotlib.colorbar as mcolorbar
from enigmatoolbox.utils.parcellation import parcel_to_surface
from enigmatoolbox.plotting import plot_cortical

plt.rcParams.update({'font.family': 'serif', 'font.serif': ['Times New Roman']})

# --- Diverging colormap ---
cmap_name = 'fig5_style'
colors_list = ['#053061', '#2166ac', '#4393c3', '#92c5de', '#f7f7f7',
               '#f4a582', '#d6604d', '#b2182b', '#67001f']
fig5_cmap = LinearSegmentedColormap.from_list(cmap_name, colors_list)
try:
    plt.colormaps.register(cmap=fig5_cmap, name=cmap_name, force=True)
except Exception:
    pass

SAVE_DIR = 'Fig5'
SUB_DIR = os.path.join(SAVE_DIR, 'SubA')
os.makedirs(SUB_DIR, exist_ok=True)

# 10 core ASD genes
asd_genes = ['SHANK3', 'SCN2A', 'MECP2', 'CHD8', 'PTEN',
             'SYNGAP1', 'NRXN1', 'SCN1A', 'ARID1B', 'ADNP']

# DK68 labels in enigmatoolbox fsa5 order
dk68_labels = [
    'lh_bankssts', 'lh_caudalanteriorcingulate', 'lh_caudalmiddlefrontal', 'lh_cuneus',
    'lh_entorhinal', 'lh_fusiform', 'lh_inferiorparietal', 'lh_inferiortemporal',
    'lh_isthmuscingulate', 'lh_lateraloccipital', 'lh_lateralorbitofrontal', 'lh_lingual',
    'lh_medialorbitofrontal', 'lh_middletemporal', 'lh_parahippocampal', 'lh_paracentral',
    'lh_parsopercularis', 'lh_parsorbitalis', 'lh_parstriangularis', 'lh_pericalcarine',
    'lh_postcentral', 'lh_posteriorcingulate', 'lh_precentral', 'lh_precuneus',
    'lh_rostralanteriorcingulate', 'lh_rostralmiddlefrontal', 'lh_superiorfrontal',
    'lh_superiorparietal', 'lh_superiortemporal', 'lh_supramarginal', 'lh_frontalpole',
    'lh_temporalpole', 'lh_transversetemporal', 'lh_insula',
    'rh_bankssts', 'rh_caudalanteriorcingulate', 'rh_caudalmiddlefrontal', 'rh_cuneus',
    'rh_entorhinal', 'rh_fusiform', 'rh_inferiorparietal', 'rh_inferiortemporal',
    'rh_isthmuscingulate', 'rh_lateraloccipital', 'rh_lateralorbitofrontal', 'rh_lingual',
    'rh_medialorbitofrontal', 'rh_middletemporal', 'rh_parahippocampal', 'rh_paracentral',
    'rh_parsopercularis', 'rh_parsorbitalis', 'rh_parstriangularis', 'rh_pericalcarine',
    'rh_postcentral', 'rh_posteriorcingulate', 'rh_precentral', 'rh_precuneus',
    'rh_rostralanteriorcingulate', 'rh_rostralmiddlefrontal', 'rh_superiorfrontal',
    'rh_superiorparietal', 'rh_superiortemporal', 'rh_supramarginal', 'rh_frontalpole',
    'rh_temporalpole', 'rh_transversetemporal', 'rh_insula'
]
label_to_idx = {label: i for i, label in enumerate(dk68_labels)}

# --- Load data and compute the global color scale ---
input_csv = os.path.join(SUB_DIR, 'SubA_ABIDE2_Gene.csv')
print(f"Reading data from: {input_csv}")
df = pd.read_csv(input_csv)
r_cols = [f"{gene}_R" for gene in asd_genes]
all_r_values = df[r_cols].values
global_abs_max = np.nanmax(np.abs(all_r_values))
global_abs_max = np.ceil(global_abs_max * 20) / 20 if global_abs_max > 0 else 0.6
print(f"Global absolute max R value: {global_abs_max}")

# --- Render each gene to a temporary PNG ---
temp_files = {}
for gene in asd_genes:
    val_col = f"{gene}_R"
    values_68 = np.full(68, np.nan)
    for _, row in df.iterrows():
        lh_label = f"lh_{row['Seed_Region']}"
        if lh_label in label_to_idx:
            values_68[label_to_idx[lh_label]] = row[val_col]
    values_fsa5 = parcel_to_surface(values_68, 'aparc_fsa5')
    temp_path = os.path.join(SUB_DIR, f"temp_{gene}.png")
    plot_cortical(array_name=values_fsa5, surface_name="fsa5", size=(1200, 300),
                  cmap=cmap_name, color_bar=False, color_range=(-global_abs_max, global_abs_max),
                  screenshot=True, filename=temp_path, background=(1, 1, 1), scale=(3, 3))
    temp_files[gene] = temp_path

# --- 5x2 grid with a shared colorbar (left-hemisphere views only) ---
fig, axes = plt.subplots(5, 2, figsize=(10, 14))
axes = axes.flatten()
for idx, gene in enumerate(asd_genes):
    ax = axes[idx]
    img = plt.imread(temp_files[gene])
    h, w, _ = img.shape
    ax.imshow(img[:, :w//2, :])
    ax.axis('off')
    ax.set_title(gene, fontsize=20, fontweight='bold', pad=2, fontstyle='italic')
for i in range(len(asd_genes), len(axes)):
    axes[i].axis('off')

cbar_ax = fig.add_axes([0.92, 0.15, 0.015, 0.7])
norm = plt.Normalize(vmin=-global_abs_max, vmax=global_abs_max)
cb = mcolorbar.ColorbarBase(cbar_ax, cmap=fig5_cmap, norm=norm, orientation='vertical')
cb.set_ticks([-global_abs_max, 0, global_abs_max])
cb.ax.tick_params(labelsize=16)
plt.subplots_adjust(left=0.02, right=0.90, hspace=0.02, wspace=0.02)

final_path = os.path.join(SAVE_DIR, "SubA.png")
plt.savefig(final_path, bbox_inches='tight', dpi=300)
plt.show()
print(f"Composite saved: {final_path}")

for f in temp_files.values():
    if os.path.exists(f):
        os.remove(f)

## Panel B — Core ASD genes: epicenter vs. non-epicenter spatial correlations

In [ ]:
"""Panel B: spatial correlation (r) of each of the 10 core ASD genes per region,
highlighting the cACC epicenter, with group means and a zero reference line."""
import os
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

# --- Global style configuration ---
GLOBAL_CONFIG = {
    'DPI': 300, 'FIGURE_WIDTH': 14, 'FIGURE_HEIGHT': 7,
    'FONT_SIZE_MAIN': 20, 'FONT_SIZE_LABEL': 18,
    'FONT_SIZE_TICK': 17, 'FONT_SIZE_LEGEND': 19,
}
plt.rcParams.update({'font.size': GLOBAL_CONFIG['FONT_SIZE_TICK'],
                     'axes.titlesize': GLOBAL_CONFIG['FONT_SIZE_MAIN'],
                     'axes.labelsize': GLOBAL_CONFIG['FONT_SIZE_LABEL'],
                     'xtick.labelsize': GLOBAL_CONFIG['FONT_SIZE_TICK'],
                     'ytick.labelsize': GLOBAL_CONFIG['FONT_SIZE_TICK'],
                     'legend.fontsize': GLOBAL_CONFIG['FONT_SIZE_LEGEND'],
                     'font.family': 'serif', 'font.serif': ['Times New Roman'],
                     'axes.unicode_minus': False})

COLOR_BG = '#ABB2B9'
COLOR_CACC = '#E74C3C'
COLOR_MEAN = '#2C3E50'
COLOR_ZERO = '#7F8C8D'

OUTPUT_DIR = 'Fig5/SubA'
ASD_GENES = ['SHANK3', 'SCN2A', 'MECP2', 'CHD8', 'SYNGAP1',
             'PTEN', 'NRXN1', 'SCN1A', 'ARID1B', 'ADNP']


def prepare_scatter_data(output_dir):
    """Gather per-gene spatial correlations for all regions across cohorts."""
    records = []
    for center in ['ABIDE2', 'CABIC']:
        fpath = os.path.join(output_dir, f'SubA_{center}_Gene.csv')
        if not os.path.exists(fpath):
            print(f"  ! Not found: {fpath}")
            continue
        df = pd.read_csv(fpath)
        for gene in ASD_GENES:
            col = f'{gene}_R'
            if col not in df.columns:
                continue
            for _, row in df.iterrows():
                records.append({'Dataset': center, 'Gene': gene, 'Region': row['Seed_Region'],
                                'R_Value': row[col],
                                'Is_cACC': row['Seed_Region'] == 'caudalanteriorcingulate',
                                'Is_Epi': row['Is_Epicenter'] == 'Yes'})
    return pd.DataFrame(records)


def plot_panel_b_scatter(output_dir):
    """Draw the Panel B scatter for both cohorts."""
    df_plot = prepare_scatter_data(output_dir)
    if df_plot.empty:
        print("No data available.")
        return
    genes = ASD_GENES
    n_genes = len(genes)
    gene_pos = {g: i for i, g in enumerate(genes)}
    datasets = ['ABIDE2', 'CABIC']

    fig, axes = plt.subplots(1, 2, figsize=(GLOBAL_CONFIG['FIGURE_WIDTH'],
                                            GLOBAL_CONFIG['FIGURE_HEIGHT']), sharey=True)
    np.random.seed(42)

    for ax_idx, (ax, center) in enumerate(zip(axes, datasets)):
        sub = df_plot[df_plot['Dataset'] == center]
        display_name = 'ABIDE-II' if center == 'ABIDE2' else center

        # Non-cACC regions with jitter
        normal = sub[~sub['Is_cACC']]
        for _, row in normal.iterrows():
            xpos = gene_pos[row['Gene']] + (np.random.rand() - 0.5) * 0.45
            ax.scatter(xpos, row['R_Value'], color=COLOR_BG, s=18, alpha=0.45,
                       edgecolors='none', zorder=2)
        # Per-gene group mean line
        for gene in genes:
            gdata = sub[sub['Gene'] == gene]
            mean_r = gdata['R_Value'].mean()
            xpos = gene_pos[gene]
            ax.plot([xpos - 0.32, xpos + 0.32], [mean_r, mean_r],
                    color=COLOR_MEAN, lw=2.2, solid_capstyle='round', zorder=4)
        # cACC highlighted diamonds
        cacc = sub[sub['Is_cACC']]
        for _, row in cacc.iterrows():
            xpos = gene_pos[row['Gene']]
            ax.scatter(xpos, row['R_Value'], color=COLOR_CACC, marker='D', s=80,
                       zorder=6, edgecolors='white', linewidths=0.8)
        # Zero reference line
        ax.axhline(0, color=COLOR_ZERO, lw=1.0, ls='--', alpha=0.6, zorder=1)
        # X-axis gene labels
        ax.set_xticks(range(n_genes))
        ax.set_xticklabels(genes, rotation=40, ha='right',
                           fontsize=GLOBAL_CONFIG['FONT_SIZE_TICK'], fontstyle='italic')
        ax.tick_params(axis='x', which='both', length=4, pad=3)
        # Y-axis
        if ax_idx == 0:
            ax.set_ylabel('Spatial Correlation (r)', fontsize=GLOBAL_CONFIG['FONT_SIZE_LABEL'])
        else:
            ax.set_ylabel('')
        ax.set_title(f'{display_name}', fontweight='bold', pad=10)
        ax.set_xlim(-0.6, n_genes - 0.4)
        ax.grid(axis='y', linestyle=':', alpha=0.35, zorder=0)
        sns.despine(ax=ax)
        n_regions = sub['Region'].nunique()
        ax.text(0.02, 0.97, f'n = {n_regions} regions', transform=ax.transAxes,
                fontsize=GLOBAL_CONFIG['FONT_SIZE_TICK'], va='top', ha='left', color='black')

    legend_handles = [
        mpatches.Patch(facecolor=COLOR_BG, alpha=0.6, label='All regions (n=34)'),
        plt.Line2D([0], [0], color=COLOR_MEAN, lw=2.5, label='Group mean'),
        plt.Line2D([0], [0], marker='D', color='w', markerfacecolor=COLOR_CACC,
                   markersize=9, label='cACC (Rank #1 epicenter)'),
    ]
    fig.legend(handles=legend_handles, loc='upper center', bbox_to_anchor=(0.5, 1.04),
               ncol=3, frameon=False, fontsize=GLOBAL_CONFIG['FONT_SIZE_LEGEND'])
    plt.tight_layout(rect=[0, 0, 1, 0.97])

    os.makedirs('Fig5', exist_ok=True)
    save_png = os.path.join('Fig5', 'SubB.png')
    fig.savefig(save_png, dpi=GLOBAL_CONFIG['DPI'], bbox_inches='tight')
    print(f"Panel B saved: {save_png}")
    plt.show()
    plt.close(fig)


if __name__ == '__main__':
    plot_panel_b_scatter(OUTPUT_DIR)

## Panel C — Seed-region ranking by mean ASD-gene correlation

In [ ]:
"""Panel C: horizontal bar plot ranking all 34 seed regions by their mean
ASD-gene correlation, with epicenter regions highlighted."""
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# --- Global style configuration ---
GLOBAL_CONFIG = {
    'DPI': 300, 'FIGURE_WIDTH': 14, 'FIGURE_HEIGHT': 12,
    'FONT_SIZE_MAIN': 24, 'FONT_SIZE_LABEL': 22,
    'FONT_SIZE_TICK': 22, 'FONT_SIZE_LEGEND': 22,
}
plt.rcParams.update({'font.size': GLOBAL_CONFIG['FONT_SIZE_TICK'],
                     'axes.titlesize': GLOBAL_CONFIG['FONT_SIZE_MAIN'],
                     'axes.labelsize': GLOBAL_CONFIG['FONT_SIZE_LABEL'],
                     'xtick.labelsize': GLOBAL_CONFIG['FONT_SIZE_TICK'],
                     'ytick.labelsize': 14,
                     'legend.fontsize': GLOBAL_CONFIG['FONT_SIZE_LEGEND'],
                     'font.family': 'serif', 'font.serif': ['Times New Roman'],
                     'axes.unicode_minus': False})

OUTPUT_DIR = 'Fig5/SubA'


def add_hue_logic(df):
    """Color-code bars: cACC/insula epicenters, other epicenters, non-epicenters."""
    df["Hue"] = 3
    for i in range(len(df)):
        is_epi = df.iloc[i]['Is_Epicenter']
        r_val = df.iloc[i]['Mean_ASD_Gene_R']
        if is_epi == 'Yes':
            if r_val < 0:
                df.iloc[i, df.columns.get_loc("Hue")] = 4
            elif df.iloc[i]['Seed_Region'] in ['caudalanteriorcingulate', 'insula']:
                df.iloc[i, df.columns.get_loc("Hue")] = 1
            else:
                df.iloc[i, df.columns.get_loc("Hue")] = 2
    return df


def prepare_data(output_dir):
    dfs = []
    for center in ['ABIDE2', 'CABIC']:
        fpath = os.path.join(output_dir, f'{center}_All_Seeds_Ranking.csv')
        if os.path.exists(fpath):
            df = pd.read_csv(fpath)
            df['Dataset'] = center
            dfs.append(df)
    return pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()


def plot_panel_c_ranking(output_dir):
    df_all = prepare_data(output_dir)
    if df_all.empty:
        print("No data files found.")
        return
    datasets = ['ABIDE2', 'CABIC']
    fig, axes = plt.subplots(1, 2, figsize=(GLOBAL_CONFIG['FIGURE_WIDTH'],
                                            GLOBAL_CONFIG['FIGURE_HEIGHT']), sharey=True)
    custom_palette = {1: '#1F78B4', 2: '#A6CEE3', 3: '#BDC3C7', 4: '#E31A1C'}

    for ax, center in zip(axes, datasets):
        df = df_all[df_all['Dataset'] == center].copy()
        df = df.sort_values('Mean_ASD_Gene_R', ascending=False)
        df = add_hue_logic(df)
        display_name = 'ABIDE-II' if center == 'ABIDE2' else center
        sns.barplot(x="Mean_ASD_Gene_R", y="Seed_Region", data=df, ax=ax,
                    hue="Hue", palette=custom_palette, dodge=False, saturation=0.6)
        ax.get_legend().set_visible(False)
        ax.axvline(0, color='black', linewidth=1.5)
        ax.set_title(f'{display_name}', fontweight='bold', pad=20)
        ax.set_xlabel("Mean Gene Correlation ($r$)", fontsize=GLOBAL_CONFIG['FONT_SIZE_LABEL'])
        ax.set_ylabel("")
        # Bold epicenter region labels
        for tick_label in ax.get_yticklabels():
            region_name = tick_label.get_text().strip()
            match = df[df['Seed_Region'] == region_name]
            if not match.empty:
                if match['Is_Epicenter'].values[0] == 'Yes':
                    tick_label.set_fontweight('bold')
                    tick_label.set_color('black')
                else:
                    tick_label.set_color('gray')

    sns.despine(ax=axes[0], left=True)
    sns.despine(ax=axes[1], left=True)
    plt.tight_layout()
    save_path = os.path.join('Fig5', 'SubC.png')
    plt.savefig(save_path, dpi=GLOBAL_CONFIG['DPI'], bbox_inches='tight')
    print(f"Panel C saved: {save_path}")
    plt.show()


if __name__ == '__main__':
    plot_panel_c_ranking(OUTPUT_DIR)

In [ ]:
"""Supplementary: report cACC single-gene spatial correlations and the
cross-cohort reproducibility (R²) of the mean ASD-gene correlation."""
import pandas as pd
from scipy.stats import pearsonr

# ---- Part 1: cACC single-gene spatial correlations ----
for center in ['ABIDE2', 'CABIC']:
    fpath = f'Fig5/SubA/SubA_{center}_Gene.csv'
    df = pd.read_csv(fpath)
    cacc = df[df['Seed_Region'] == 'caudalanteriorcingulate'].iloc[0]
    print(f"{'='*50}")
    print(f"  {center} — caudalanteriorcingulate (cACC)")
    print(f"{'='*50}")
    print(f"  Mean_ASD_Gene_R = {cacc['Mean_ASD_Gene_R']:.4f}")
    print(f"  NRXN1_R         = {cacc['NRXN1_R']:.4f}")
    print(f"  SCN2A_R         = {cacc['SCN2A_R']:.4f}")
    print()

# ---- Part 2: cross-cohort correlation (R²) of Mean_ASD_Gene_R ----
print(f"{'='*50}")
print("  Cross-cohort correlation of Mean_ASD_Gene_R")
print(f"{'='*50}")
df_a = pd.read_csv('Fig5/SubA/SubA_ABIDE2_Gene.csv')
df_c = pd.read_csv('Fig5/SubA/SubA_CABIC_Gene.csv')
df_merge = df_a[['Seed_Region', 'Mean_ASD_Gene_R']].merge(
    df_c[['Seed_Region', 'Mean_ASD_Gene_R']], on='Seed_Region',
    suffixes=('_ABIDE2', '_CABIC'))
r, p = pearsonr(df_merge['Mean_ASD_Gene_R_ABIDE2'], df_merge['Mean_ASD_Gene_R_CABIC'])
print(f"  r   = {r:.4f}")
print(f"  R²  = {r**2:.4f}")
print(f"  P   = {p:.4e}")
print(f"  N   = {len(df_merge)} seeds")